<a href="https://www.kaggle.com/code/luisgonzalez0822/comp-4018-as3?scriptVersionId=314615576" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv
/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv


In [2]:
import pandas as pd
import json
import sqlite3
import os

movies_df = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv')
credits_df = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv')

print(f"Movies:  {movies_df.shape}")
print(f"Credits: {credits_df.shape}")

Movies:  (4803, 20)
Credits: (4803, 4)


In [3]:
def parse_json_col(val):
    """Convierte una cadena JSON a lista Python; retorna [] si falla."""
    try:
        return json.loads(val)
    except (json.JSONDecodeError, TypeError):
        return []

JSON_COLS_MOVIES = ['genres', 'keywords', 'production_companies',
                    'production_countries', 'spoken_languages']
for col in JSON_COLS_MOVIES:
    movies_df[col] = movies_df[col].apply(parse_json_col)

credits_df['cast'] = credits_df['cast'].apply(parse_json_col)
credits_df['crew'] = credits_df['crew'].apply(parse_json_col)

credits_df = credits_df.rename(columns={'movie_id': 'id'})
df = movies_df.merge(credits_df[['id', 'cast', 'crew']], on='id', how='left')

print(f"DataFrame unificado: {df.shape}")
print("Ejemplo genres parseado:", df['genres'].iloc[0])

DataFrame unificado: (4803, 22)
Ejemplo genres parseado: [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 878, 'name': 'Science Fiction'}]


In [4]:
# Tabla: Movies 
movies_table = df[['id','title','budget','homepage','original_language',
                    'original_title','overview','popularity','release_date',
                    'revenue','runtime','status','tagline',
                    'vote_average','vote_count']].copy()
movies_table = movies_table.rename(columns={'id': 'movie_id'})
movies_table['release_date'] = pd.to_datetime(movies_table['release_date'], errors='coerce')
movies_table['release_year'] = movies_table['release_date'].dt.year.astype('Int64')
movies_table['release_date'] = movies_table['release_date'].dt.strftime('%Y-%m-%d')

# Tabla: Genres + Movie_Genres 
genres_rows, movie_genres_rows = [], []
for _, row in df.iterrows():
    for g in row['genres']:
        genres_rows.append({'genre_id': g['id'], 'genre_name': g['name']})
        movie_genres_rows.append({'movie_id': row['id'], 'genre_id': g['id']})

genres_table       = pd.DataFrame(genres_rows).drop_duplicates('genre_id').reset_index(drop=True)
movie_genres_table = pd.DataFrame(movie_genres_rows).drop_duplicates().reset_index(drop=True)

# Tabla: Keywords + Movie_Keywords 
kw_rows, movie_kw_rows = [], []
for _, row in df.iterrows():
    for k in row['keywords']:
        kw_rows.append({'keyword_id': k['id'], 'keyword_name': k['name']})
        movie_kw_rows.append({'movie_id': row['id'], 'keyword_id': k['id']})

keywords_table       = pd.DataFrame(kw_rows).drop_duplicates('keyword_id').reset_index(drop=True)
movie_keywords_table = pd.DataFrame(movie_kw_rows).drop_duplicates().reset_index(drop=True)

# Tabla: Production_Companies + Movie_Companies 
comp_rows, movie_comp_rows = [], []
for _, row in df.iterrows():
    for c in row['production_companies']:
        comp_rows.append({'company_id': c['id'], 'company_name': c['name']})
        movie_comp_rows.append({'movie_id': row['id'], 'company_id': c['id']})

companies_table       = pd.DataFrame(comp_rows).drop_duplicates('company_id').reset_index(drop=True)
movie_companies_table = pd.DataFrame(movie_comp_rows).drop_duplicates().reset_index(drop=True)

# Tabla: Production_Countries + Movie_Countries 
country_rows, movie_country_rows = [], []
for _, row in df.iterrows():
    for c in row['production_countries']:
        country_rows.append({'country_iso': c['iso_3166_1'], 'country_name': c['name']})
        movie_country_rows.append({'movie_id': row['id'], 'country_iso': c['iso_3166_1']})

countries_table       = pd.DataFrame(country_rows).drop_duplicates('country_iso').reset_index(drop=True)
movie_countries_table = pd.DataFrame(movie_country_rows).drop_duplicates().reset_index(drop=True)

# Tabla: Spoken_Languages + Movie_Languages 
lang_rows, movie_lang_rows = [], []
for _, row in df.iterrows():
    for l in row['spoken_languages']:
        lang_rows.append({'language_iso': l['iso_639_1'], 'language_name': l['name']})
        movie_lang_rows.append({'movie_id': row['id'], 'language_iso': l['iso_639_1']})

languages_table       = pd.DataFrame(lang_rows).drop_duplicates('language_iso').reset_index(drop=True)
movie_languages_table = pd.DataFrame(movie_lang_rows).drop_duplicates().reset_index(drop=True)

# Tabla: Persons + Cast + Crew 
persons_rows, cast_rows, crew_rows = [], [], []
for _, row in df.iterrows():
    movie_id = row['id']
    for c in row['cast']:
        persons_rows.append({'person_id': c['id'], 'person_name': c['name'], 'gender': c.get('gender')})
        cast_rows.append({'movie_id': movie_id, 'person_id': c['id'],
                          'character': c.get('character',''), 'cast_order': c.get('order'),
                          'credit_id': c.get('credit_id','')})
    for cr in row['crew']:
        persons_rows.append({'person_id': cr['id'], 'person_name': cr['name'], 'gender': cr.get('gender')})
        crew_rows.append({'movie_id': movie_id, 'person_id': cr['id'],
                          'department': cr.get('department',''), 'job': cr.get('job',''),
                          'credit_id': cr.get('credit_id','')})

persons_table = pd.DataFrame(persons_rows).drop_duplicates('person_id').reset_index(drop=True)
cast_table    = pd.DataFrame(cast_rows).drop_duplicates(subset=['movie_id','person_id','character']).reset_index(drop=True)
crew_table    = pd.DataFrame(crew_rows).drop_duplicates(subset=['movie_id','person_id','job']).reset_index(drop=True)

# Resumen 
print(f"Movies:            {movies_table.shape}")
print(f"Genres:            {genres_table.shape}  | Movie_Genres:    {movie_genres_table.shape}")
print(f"Keywords:          {keywords_table.shape} | Movie_Keywords:  {movie_keywords_table.shape}")
print(f"Companies:         {companies_table.shape} | Movie_Companies: {movie_companies_table.shape}")
print(f"Countries:         {countries_table.shape}  | Movie_Countries: {movie_countries_table.shape}")
print(f"Languages:         {languages_table.shape}  | Movie_Languages: {movie_languages_table.shape}")
print(f"Persons:           {persons_table.shape} | Cast: {cast_table.shape} | Crew: {crew_table.shape}")

Movies:            (4803, 16)
Genres:            (20, 2)  | Movie_Genres:    (12160, 2)
Keywords:          (9813, 2) | Movie_Keywords:  (36194, 2)
Companies:         (5047, 2) | Movie_Companies: (13677, 2)
Countries:         (88, 2)  | Movie_Countries: (6436, 2)
Languages:         (87, 2)  | Movie_Languages: (6937, 2)
Persons:           (104842, 3) | Cast: (106257, 5) | Crew: (129581, 5)


In [5]:
DB_PATH = '/kaggle/working/tmdb_normalized.db'
conn   = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

# DDL: Crear tablas con PK y FK 
DDL = [
"""CREATE TABLE IF NOT EXISTS Movies (
    movie_id INTEGER PRIMARY KEY, title TEXT, budget REAL, homepage TEXT,
    original_language TEXT, original_title TEXT, overview TEXT, popularity REAL,
    release_date TEXT, release_year INTEGER, revenue REAL, runtime REAL,
    status TEXT, tagline TEXT, vote_average REAL, vote_count INTEGER)""",

"""CREATE TABLE IF NOT EXISTS Genres (
    genre_id INTEGER PRIMARY KEY, genre_name TEXT NOT NULL)""",

"""CREATE TABLE IF NOT EXISTS Movie_Genres (
    movie_id INTEGER, genre_id INTEGER,
    PRIMARY KEY (movie_id, genre_id),
    FOREIGN KEY (movie_id) REFERENCES Movies(movie_id),
    FOREIGN KEY (genre_id) REFERENCES Genres(genre_id))""",

"""CREATE TABLE IF NOT EXISTS Keywords (
    keyword_id INTEGER PRIMARY KEY, keyword_name TEXT NOT NULL)""",

"""CREATE TABLE IF NOT EXISTS Movie_Keywords (
    movie_id INTEGER, keyword_id INTEGER,
    PRIMARY KEY (movie_id, keyword_id),
    FOREIGN KEY (movie_id)   REFERENCES Movies(movie_id),
    FOREIGN KEY (keyword_id) REFERENCES Keywords(keyword_id))""",

"""CREATE TABLE IF NOT EXISTS Production_Companies (
    company_id INTEGER PRIMARY KEY, company_name TEXT NOT NULL)""",

"""CREATE TABLE IF NOT EXISTS Movie_Companies (
    movie_id INTEGER, company_id INTEGER,
    PRIMARY KEY (movie_id, company_id),
    FOREIGN KEY (movie_id)   REFERENCES Movies(movie_id),
    FOREIGN KEY (company_id) REFERENCES Production_Companies(company_id))""",

"""CREATE TABLE IF NOT EXISTS Production_Countries (
    country_iso TEXT PRIMARY KEY, country_name TEXT NOT NULL)""",

"""CREATE TABLE IF NOT EXISTS Movie_Countries (
    movie_id INTEGER, country_iso TEXT,
    PRIMARY KEY (movie_id, country_iso),
    FOREIGN KEY (movie_id)    REFERENCES Movies(movie_id),
    FOREIGN KEY (country_iso) REFERENCES Production_Countries(country_iso))""",

"""CREATE TABLE IF NOT EXISTS Spoken_Languages (
    language_iso TEXT PRIMARY KEY, language_name TEXT NOT NULL)""",

"""CREATE TABLE IF NOT EXISTS Movie_Languages (
    movie_id INTEGER, language_iso TEXT,
    PRIMARY KEY (movie_id, language_iso),
    FOREIGN KEY (movie_id)     REFERENCES Movies(movie_id),
    FOREIGN KEY (language_iso) REFERENCES Spoken_Languages(language_iso))""",

"""CREATE TABLE IF NOT EXISTS Persons (
    person_id INTEGER PRIMARY KEY, person_name TEXT NOT NULL, gender INTEGER)""",

"""CREATE TABLE IF NOT EXISTS Cast (
    movie_id INTEGER, person_id INTEGER, character TEXT,
    cast_order INTEGER, credit_id TEXT,
    PRIMARY KEY (movie_id, person_id, character),
    FOREIGN KEY (movie_id)  REFERENCES Movies(movie_id),
    FOREIGN KEY (person_id) REFERENCES Persons(person_id))""",

"""CREATE TABLE IF NOT EXISTS Crew (
    movie_id INTEGER, person_id INTEGER, department TEXT,
    job TEXT, credit_id TEXT,
    PRIMARY KEY (movie_id, person_id, job),
    FOREIGN KEY (movie_id)  REFERENCES Movies(movie_id),
    FOREIGN KEY (person_id) REFERENCES Persons(person_id))"""
]

for stmt in DDL:
    cursor.execute(stmt)
conn.commit()
print("✔ Esquema creado")

# Cargar datos
TABLE_MAP = {
    'Movies':               movies_table,
    'Genres':               genres_table,
    'Movie_Genres':         movie_genres_table,
    'Keywords':             keywords_table,
    'Movie_Keywords':       movie_keywords_table,
    'Production_Companies': companies_table,
    'Movie_Companies':      movie_companies_table,
    'Production_Countries': countries_table,
    'Movie_Countries':      movie_countries_table,
    'Spoken_Languages':     languages_table,
    'Movie_Languages':      movie_languages_table,
    'Persons':              persons_table,
    'Cast':                 cast_table,
    'Crew':                 crew_table,
}

for name, table in TABLE_MAP.items():
    table.to_sql(name, conn, if_exists='replace', index=False)
    count = cursor.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"  ✔ {name:25s} → {count:,} filas")

conn.commit()
print(f"\n✔ BD guardada en: {DB_PATH}")

✔ Esquema creado
  ✔ Movies                    → 4,803 filas
  ✔ Genres                    → 20 filas
  ✔ Movie_Genres              → 12,160 filas
  ✔ Keywords                  → 9,813 filas
  ✔ Movie_Keywords            → 36,194 filas
  ✔ Production_Companies      → 5,047 filas
  ✔ Movie_Companies           → 13,677 filas
  ✔ Production_Countries      → 88 filas
  ✔ Movie_Countries           → 6,436 filas
  ✔ Spoken_Languages          → 87 filas
  ✔ Movie_Languages           → 6,937 filas
  ✔ Persons                   → 104,842 filas
  ✔ Cast                      → 106,257 filas
  ✔ Crew                      → 129,581 filas

✔ BD guardada en: /kaggle/working/tmdb_normalized.db


In [6]:
KEYWORD = "space"  # ← cambia la palabra aquí

q1 = """
SELECT
    m.title                                  AS Titulo,
    GROUP_CONCAT(DISTINCT g.genre_name)      AS Generos,
    GROUP_CONCAT(DISTINCT sl.language_name)  AS Idiomas,
    ROUND(m.popularity, 2)                   AS Popularidad,
    GROUP_CONCAT(DISTINCT p.person_name)     AS Cast_Principal
FROM Movies m
LEFT JOIN Movie_Genres    mg ON m.movie_id    = mg.movie_id
LEFT JOIN Genres           g ON mg.genre_id   = g.genre_id
LEFT JOIN Movie_Languages ml ON m.movie_id    = ml.movie_id
LEFT JOIN Spoken_Languages sl ON ml.language_iso = sl.language_iso
LEFT JOIN Cast             c ON m.movie_id    = c.movie_id
LEFT JOIN Persons          p ON c.person_id   = p.person_id
WHERE m.title LIKE :kw OR m.overview LIKE :kw
GROUP BY m.movie_id
ORDER BY m.popularity DESC
LIMIT 15;
"""
pd.read_sql_query(q1, conn, params={'kw': f'%{KEYWORD}%'})

,Titulo,Generos,Idiomas,Popularidad,Cast_Principal
0,Interstellar,"Adventure,Drama,Science Fiction",English,724.25,"Russ Fega,Anne Hathaway,Matt Damon,Casey Affle..."
1,Tomorrowland,"Adventure,Science Fiction,Mystery,Family",English,130.31,"George Clooney,Chris Bauer,Michael Giacchino,K..."
2,Gravity,"Drama,Thriller,Science Fiction",English,110.15,"Ed Harris,George Clooney,Sandra Bullock,Phaldu..."
3,Alien,"Horror,Action,Thriller,Science Fiction","English,Español",94.18,"Ian Holm,Tom Skerritt,Veronica Cartwright,Harr..."
4,2001: A Space Odyssey,"Adventure,Science Fiction,Mystery","English,Pусский",86.20,"Keir Dullea,Gary Lockwood,William Sylvester,Da..."
5,Gattaca,"Thriller,Science Fiction,Mystery,Romance","English,Esperanto",70.40,"Uma Thurman,Ethan Hawke,Alan Arkin,Xander Berk..."
6,Oblivion,"Adventure,Action,Science Fiction,Mystery",English,67.70,"Morgan Freeman,Tom Cruise,Melissa Leo,Nikolaj ..."
7,Elysium,"Drama,Action,Thriller,Science Fiction",English,67.34,"William Fichtner,Jodie Foster,Matt Damon,Alice..."
8,WALL·E,"Animation,Family",English,66.39,"Andrew Stanton,Ben Burtt,Jeff Pidgeon,John Rat..."
9,Star Trek Beyond,"Adventure,Action,Science Fiction",English,65.35,"Deep Roy,Karl Urban,Zoe Saldana,Simon Pegg,Gre..."


In [7]:
YEAR = 2015  # ← cambia el año aquí

q2 = """
SELECT
    m.title                 AS Titulo,
    m.release_year          AS Ano,
    ROUND(m.popularity, 2)  AS Popularidad,
    ROUND(m.vote_average,2) AS Promedio_Votos
FROM Movies m
WHERE m.release_year >= :year
ORDER BY m.release_year ASC, m.title ASC
LIMIT 15;
"""
pd.read_sql_query(q2, conn, params={'year': YEAR})

,Titulo,Ano,Popularidad,Promedio_Votos
0,#Horror,2015,2.82,3.3
1,10 Days in a Madhouse,2015,0.49,4.3
2,90 Minutes in Heaven,2015,3.94,5.4
3,A LEGO Brickumentary,2015,1.63,6.4
4,A Tale of Three Cities,2015,0.75,6.3
5,AWOL-72,2015,1.14,2.8
6,Abandoned,2015,3.07,5.8
7,Accidental Love,2015,5.64,3.9
8,Aloha,2015,29.65,5.2
9,Alvin and the Chipmunks: The Road Chip,2015,27.87,5.8


In [8]:
q3 = """
SELECT
    g.genre_name                   AS Genero,
    COUNT(DISTINCT mg.movie_id)    AS Num_Peliculas,
    ROUND(AVG(m.vote_average), 2)  AS Promedio_Votos,
    ROUND(AVG(m.popularity),   2)  AS Popularidad_Promedio
FROM Genres g
JOIN Movie_Genres mg ON g.genre_id  = mg.genre_id
JOIN Movies        m ON mg.movie_id = m.movie_id
GROUP BY g.genre_id
HAVING AVG(m.popularity) >= 6
ORDER BY Popularidad_Promedio DESC
LIMIT 10;
"""
pd.read_sql_query(q3, conn)

,Genero,Num_Peliculas,Promedio_Votos,Popularidad_Promedio
0,Adventure,790,6.16,39.27
1,Animation,234,6.34,38.81
2,Science Fiction,535,6.01,36.45
3,Fantasy,424,6.10,36.39
4,Action,1154,5.99,30.94
5,Family,513,6.03,27.83
6,Mystery,348,6.18,24.59
7,Thriller,1274,6.01,24.46
8,War,144,6.71,23.78
9,Crime,696,6.27,22.85


In [9]:
q4 = """
SELECT
    m.release_year                 AS Ano,
    COUNT(DISTINCT m.movie_id)     AS Num_Peliculas,
    ROUND(AVG(m.vote_average), 2)  AS Promedio_Votos,
    ROUND(AVG(m.popularity),   2)  AS Popularidad_Promedio
FROM Movies m
WHERE m.release_year IS NOT NULL
GROUP BY m.release_year
HAVING AVG(m.popularity) >= 6
ORDER BY Popularidad_Promedio DESC;
"""
pd.read_sql_query(q4, conn)

,Ano,Num_Peliculas,Promedio_Votos,Popularidad_Promedio
0,1975,6,7.30,47.49
1,1942,2,7.35,45.69
2,1939,3,7.67,42.89
3,1937,2,7.65,42.10
4,1957,2,7.95,41.18
...,...,...,...,...
69,1930,1,6.10,8.48
70,1951,3,7.17,8.15
71,1941,1,6.80,7.70
72,1970,12,6.65,7.05
